# TCC2 — Dengue Forecasting: Results & Analysis (Distrito Federal)

Comprehensive analysis of XGBoost model performance, feature importance (SHAP), and temporal validation for the TCC2 thesis.

**Authors:** Pedro Lucas Santana & Thiago Ribeiro Freitas  
**Program:** Software Engineering — University of Brasília (UnB)  
**Municipality:** Brasília / Distrito Federal (IBGE 5300108)  

This notebook loads trained models and precomputed metrics from Notebook 2, then generates all figures and tables required for the thesis document.

---
## 1. Setup

In [ ]:
!pip install -q xgboost shap huggingface_hub pyarrow

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import xgboost as xgb
import shap
import json
import warnings
from pathlib import Path
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    roc_auc_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report
)

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# ── Paths ──────────────────────────────────────────────────────────────────
# Notebook 2 output added as Kaggle dataset
ARTIFACTS_DIR = Path('/kaggle/input/tcc2-treino-artifacts')
# Fallback to /kaggle/working if running after notebook 2 in same session
if not ARTIFACTS_DIR.exists():
    ARTIFACTS_DIR = Path('/kaggle/working')

OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HF_REPO = 'pedrolucassantanaf/dengue-tcc2-data'
HF_PATH = 'data/model_ready/test.parquet'
IBGE_DF = '5300108'

CLASS_LABELS = {0: 'Low', 1: 'Medium', 2: 'High', 3: 'Outbreak'}
CLASS_NAMES  = ['Low', 'Medium', 'High', 'Outbreak']

# ── Style ─────────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'legend.fontsize': 10,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.facecolor': 'white',
    'savefig.bbox': 'tight',
})

# Color palette — consistent throughout
COLORS = {
    'primary':   '#2171B5',   # strong blue
    'secondary': '#CB181D',   # strong red
    'accent':    '#238B45',   # green
    'highlight': '#D94801',   # orange
    'muted':     '#969696',   # gray
    'light':     '#C6DBEF',   # light blue
    'bg_fill':   '#DEEBF7',   # very light blue
}

def save_fig(fig, name):
    """Save figure as high-resolution PNG for thesis inclusion."""
    path = OUTPUT_DIR / f'fig_{name}.png'
    fig.savefig(str(path), dpi=300, bbox_inches='tight')
    print(f'  Saved: {path}')

print(f'Artifacts directory: {ARTIFACTS_DIR}')
print(f'Output directory:    {OUTPUT_DIR}')
print('Setup complete.')

---
## 2. Load Artifacts

In [ ]:
# ── Load models ───────────────────────────────────────────────────────────
reg_baseline = xgb.XGBRegressor()
reg_baseline.load_model(str(ARTIFACTS_DIR / 'xgb_reg_baseline.ubj'))

reg_tuned = xgb.XGBRegressor()
reg_tuned.load_model(str(ARTIFACTS_DIR / 'xgb_reg_tuned.ubj'))

reg_sinan_only = xgb.XGBRegressor()
reg_sinan_only.load_model(str(ARTIFACTS_DIR / 'xgb_reg_sinan_only.ubj'))

cls_baseline = xgb.XGBClassifier()
cls_baseline.load_model(str(ARTIFACTS_DIR / 'xgb_cls_baseline.ubj'))

cls_tuned = xgb.XGBClassifier()
cls_tuned.load_model(str(ARTIFACTS_DIR / 'xgb_cls_tuned.ubj'))

print('Models loaded:')
for name in ['reg_baseline', 'reg_tuned', 'reg_sinan_only', 'cls_baseline', 'cls_tuned']:
    print(f'  {name}')

# ── Load metrics ──────────────────────────────────────────────────────────
metrics_all = pd.read_csv(ARTIFACTS_DIR / 'metrics_all.csv')
wf_reg = pd.read_csv(ARTIFACTS_DIR / 'walk_forward_reg.csv')
wf_cls = pd.read_csv(ARTIFACTS_DIR / 'walk_forward_cls.csv')
optuna_reg = pd.read_csv(ARTIFACTS_DIR / 'optuna_reg_trials.csv')
optuna_cls = pd.read_csv(ARTIFACTS_DIR / 'optuna_cls_trials.csv')

print(f'\nMetrics table:  {len(metrics_all)} experiments')
print(f'Walk-forward:   {len(wf_reg)} years (reg), {len(wf_cls)} years (cls)')
print(f'Optuna trials:  {len(optuna_reg)} (reg), {len(optuna_cls)} (cls)')

# ── Load test data ────────────────────────────────────────────────────────
X_test = pd.read_parquet(ARTIFACTS_DIR / 'X_test.parquet')
y_test_reg = pd.read_parquet(ARTIFACTS_DIR / 'y_test_reg.parquet').squeeze()
y_test_cls = pd.read_parquet(ARTIFACTS_DIR / 'y_test_cls.parquet').squeeze()

with open(ARTIFACTS_DIR / 'feature_names.json', 'r') as f:
    feature_names = json.load(f)

print(f'\nTest set: {len(X_test)} rows, {len(feature_names)} features')
print(f'Regression target (notificacoes_t4): min={y_test_reg.min():.0f}, max={y_test_reg.max():.0f}')
print(f'Classification target (risco_surto_t4) distribution:')
for cls_val, cls_name in CLASS_LABELS.items():
    count = (y_test_cls == cls_val).sum()
    print(f'  {cls_name}: {count}')

In [ ]:
# ── Load full test.parquet from HuggingFace for time context ─────────────
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient

try:
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret('HF_TOKEN')
except Exception:
    import os
    hf_token = os.environ.get('HF_TOKEN', None)

hf_test_path = hf_hub_download(
    repo_id=HF_REPO,
    filename=HF_PATH,
    repo_type='dataset',
    token=hf_token,
)

test_full = pd.read_parquet(hf_test_path)
test_df = test_full[test_full['ibge_municipio'].astype(str) == IBGE_DF].copy()
test_df = test_df.sort_values(['ano', 'semana_epidemiologica']).reset_index(drop=True)

print(f'HF test.parquet loaded: {len(test_df)} rows for DF')
print(f'Period: {test_df["ano"].min()}-{test_df["ano"].max()}')
print(f'Columns available: ano, semana_epidemiologica, notificacoes_t4, ...')

# Also load train for distribution shift analysis
hf_train_path = hf_hub_download(
    repo_id=HF_REPO,
    filename='data/model_ready/train.parquet',
    repo_type='dataset',
    token=hf_token,
)
hf_val_path = hf_hub_download(
    repo_id=HF_REPO,
    filename='data/model_ready/val.parquet',
    repo_type='dataset',
    token=hf_token,
)

train_full = pd.read_parquet(hf_train_path)
val_full = pd.read_parquet(hf_val_path)

train_df = train_full[train_full['ibge_municipio'].astype(str) == IBGE_DF].copy()
val_df = val_full[val_full['ibge_municipio'].astype(str) == IBGE_DF].copy()

# Combine train+val as was done during training
trainval_df = pd.concat([train_df, val_df], ignore_index=True)
trainval_df = trainval_df.sort_values(['ano', 'semana_epidemiologica']).reset_index(drop=True)

print(f'\nTrain+Val from HF: {len(trainval_df)} rows for DF')
print(f'  Train: {len(train_df)} rows (< 2022)')
print(f'  Val:   {len(val_df)} rows (2022-2023)')

del train_full, val_full, test_full

In [ ]:
# ── Generate predictions from loaded models ──────────────────────────────

# Regression predictions (tuned model — primary)
y_pred_log_tuned = reg_tuned.predict(X_test)
y_pred_tuned = np.expm1(np.maximum(y_pred_log_tuned, 0))

# Regression predictions (baseline)
y_pred_log_baseline = reg_baseline.predict(X_test)
y_pred_baseline = np.expm1(np.maximum(y_pred_log_baseline, 0))

# Regression predictions (SINAN-only)
# Identify climate feature columns
climate_keywords = ['temp_', 'rain_', 'humidity_', 'pressure_', 'wind_', 'radiation_']
sinan_only_cols = [c for c in feature_names if not any(k in c for k in climate_keywords)]
y_pred_log_sinan = reg_sinan_only.predict(X_test[sinan_only_cols])
y_pred_sinan = np.expm1(np.maximum(y_pred_log_sinan, 0))

# Classification predictions (tuned model)
y_pred_cls_tuned = cls_tuned.predict(X_test)
y_pred_proba_tuned = cls_tuned.predict_proba(X_test)

# Log-transformed test target
y_test_log = np.log1p(y_test_reg)

print('Predictions generated:')
print(f'  Reg tuned:     R2_log={r2_score(y_test_log, y_pred_log_tuned):.4f}, '
      f'R2_orig={r2_score(y_test_reg, y_pred_tuned):.4f}')
print(f'  Reg baseline:  R2_log={r2_score(y_test_log, y_pred_log_baseline):.4f}, '
      f'R2_orig={r2_score(y_test_reg, y_pred_baseline):.4f}')
print(f'  Reg SINAN-only:R2_log={r2_score(y_test_log, y_pred_log_sinan):.4f}, '
      f'R2_orig={r2_score(y_test_reg, y_pred_sinan):.4f}')
print(f'  Cls tuned:     F1_macro={f1_score(y_test_cls, y_pred_cls_tuned, average="macro", zero_division=0):.4f}')

---
## 3. Experiment Comparison Table

Summary of all experiments: baseline vs Optuna-tuned, SINAN-only vs SINAN+INMET, regression and classification.

In [ ]:
# ── Styled experiment comparison table ────────────────────────────────────
display_metrics = metrics_all.copy()

# Rename for display
col_rename = {
    'experiment': 'Experiment',
    'mae': 'MAE',
    'rmse': 'RMSE',
    'r2_log': 'R² (log)',
    'r2_original': 'R² (original)',
    'auc_macro': 'AUC (macro)',
    'f1_macro': 'F1 (macro)',
    'duration_s': 'Time (s)',
}
display_metrics = display_metrics.rename(columns=col_rename)

# Identify best values: for MAE/RMSE lower is better; for R²/AUC/F1 higher is better
higher_better = ['R² (log)', 'R² (original)', 'AUC (macro)', 'F1 (macro)']
lower_better  = ['MAE', 'RMSE', 'Time (s)']

def highlight_best(s):
    """Highlight best value in each numeric column."""
    styles = [''] * len(s)
    numeric_vals = pd.to_numeric(s, errors='coerce')
    if numeric_vals.notna().sum() == 0:
        return styles
    if s.name in higher_better:
        best_idx = numeric_vals.idxmax()
    elif s.name in lower_better:
        best_idx = numeric_vals.idxmin()
    else:
        return styles
    if pd.notna(best_idx):
        styles[best_idx] = 'font-weight: bold; background-color: #d4edda;'
    return styles

styled = (
    display_metrics
    .style
    .apply(highlight_best, axis=0)
    .format({
        'MAE': '{:.1f}',
        'RMSE': '{:.1f}',
        'R² (log)': '{:.4f}',
        'R² (original)': '{:.4f}',
        'AUC (macro)': '{:.4f}',
        'F1 (macro)': '{:.4f}',
        'Time (s)': '{:.1f}',
    }, na_rep='—')
    .set_caption('Table 1: Experiment Comparison — Dengue Forecasting Models (Distrito Federal)')
    .set_table_styles([
        {'selector': 'caption', 'props': [('font-size', '14px'), ('font-weight', 'bold'), ('padding', '10px')]},
        {'selector': 'th', 'props': [('background-color', '#f0f0f0'), ('text-align', 'center')]},
        {'selector': 'td', 'props': [('text-align', 'center')]},
    ])
)

styled

**Interpretation:**

- The **Optuna-tuned regression model** (SINAN+INMET) achieves the best R² on log-transformed predictions, indicating it best captures the *relative magnitude* of dengue notifications 4 weeks ahead.
- R² on the **original scale** is low for all models due to the extreme distribution shift in 2024 (see Section 9). This is expected and does not indicate model failure — it reflects the unprecedented nature of the 2024 outbreak.
- The **SINAN+INMET** combination outperforms **SINAN-only**, confirming that climate data from station A001 (1.18 km from Brasilia center) adds predictive value.
- Classification F1 scores reflect the difficulty of distinguishing four risk levels under distribution shift.

---
## 4. Regression — Predicted vs Actual (Scatter)

In [ ]:
r2_log_val = r2_score(y_test_log, y_pred_log_tuned)
r2_orig_val = r2_score(y_test_reg, y_pred_tuned)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Left: Log scale ──────────────────────────────────────────────────────
ax = axes[0]
# Color points by actual magnitude
magnitude = y_test_reg.values
scatter = ax.scatter(
    y_test_log, y_pred_log_tuned,
    c=magnitude, cmap='YlOrRd', s=30, alpha=0.7,
    edgecolors='gray', linewidths=0.3,
    norm=plt.Normalize(vmin=0, vmax=np.percentile(magnitude, 95))
)
lim_log = max(y_test_log.max(), y_pred_log_tuned.max()) * 1.05
ax.plot([0, lim_log], [0, lim_log], '--', color=COLORS['secondary'], linewidth=1.5,
        label='Perfect prediction', zorder=5)
ax.set_xlabel('Actual — log1p(notifications)')
ax.set_ylabel('Predicted — log1p(notifications)')
ax.set_title(f'(a) Log Scale — R² = {r2_log_val:.3f}')
ax.legend(loc='upper left', framealpha=0.9)
ax.set_xlim(0, lim_log)
ax.set_ylim(0, lim_log)
ax.set_aspect('equal', adjustable='box')

# ── Right: Original scale ────────────────────────────────────────────────
ax = axes[1]
scatter2 = ax.scatter(
    y_test_reg, y_pred_tuned,
    c=magnitude, cmap='YlOrRd', s=30, alpha=0.7,
    edgecolors='gray', linewidths=0.3,
    norm=plt.Normalize(vmin=0, vmax=np.percentile(magnitude, 95))
)
lim_orig = max(y_test_reg.max(), max(y_pred_tuned)) * 1.05
ax.plot([0, lim_orig], [0, lim_orig], '--', color=COLORS['secondary'], linewidth=1.5,
        label='Perfect prediction', zorder=5)
ax.set_xlabel('Actual — notifications')
ax.set_ylabel('Predicted — notifications')
ax.set_title(f'(b) Original Scale — R² = {r2_orig_val:.3f}')
ax.legend(loc='upper left', framealpha=0.9)

cbar = fig.colorbar(scatter2, ax=axes, shrink=0.8, pad=0.02)
cbar.set_label('Actual notifications (original scale)', fontsize=10)

fig.suptitle('Predicted vs Actual Dengue Notifications — Tuned XGBoost Regression (Test 2024)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig(fig, 'scatter_pred_vs_actual')
plt.show()

---
## 5. Regression — Time Series (Test Period)

Actual vs predicted dengue notifications over the 2024 test period, week by week.

In [ ]:
# Build time axis from HF data
weeks = test_df['semana_epidemiologica'].values[:len(y_test_reg)]
actual = y_test_reg.values
predicted = y_pred_tuned

fig, ax = plt.subplots(figsize=(14, 6))

# Plot actual and predicted
ax.plot(weeks, actual, color=COLORS['primary'], linewidth=2, label='Actual', zorder=3)
ax.plot(weeks, predicted, color=COLORS['secondary'], linewidth=1.8, linestyle='--',
        label='Predicted (Tuned XGBoost)', zorder=3)

# Shade the region between
ax.fill_between(weeks, actual, predicted, alpha=0.15, color=COLORS['muted'],
                label='Prediction gap')

# Annotate peak outbreak weeks
peak_idx = np.argmax(actual)
peak_week = weeks[peak_idx]
peak_val = actual[peak_idx]
ax.annotate(
    f'Peak: {peak_val:,.0f}\n(Week {peak_week})',
    xy=(peak_week, peak_val),
    xytext=(peak_week + 5, peak_val * 0.85),
    fontsize=10, fontweight='bold', color=COLORS['secondary'],
    arrowprops=dict(arrowstyle='->', color=COLORS['secondary'], lw=1.5),
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=COLORS['secondary'], alpha=0.9),
)

# Also annotate predicted peak
pred_peak_idx = np.argmax(predicted)
pred_peak_week = weeks[pred_peak_idx]
pred_peak_val = predicted[pred_peak_idx]
ax.annotate(
    f'Pred. peak: {pred_peak_val:,.0f}\n(Week {pred_peak_week})',
    xy=(pred_peak_week, pred_peak_val),
    xytext=(pred_peak_week + 5, pred_peak_val + peak_val * 0.10),
    fontsize=9, color=COLORS['primary'],
    arrowprops=dict(arrowstyle='->', color=COLORS['primary'], lw=1.2),
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=COLORS['primary'], alpha=0.9),
)

ax.set_xlabel('Epidemiological Week (2024)', fontsize=12)
ax.set_ylabel('Dengue Notifications (t+4)', fontsize=12)
ax.set_title('Test Period (2024): Actual vs Predicted Dengue Notifications',
             fontsize=14, fontweight='bold')
ax.legend(loc='upper right', framealpha=0.9, fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
save_fig(fig, 'timeseries_actual_vs_predicted')
plt.show()

---
## 6. SHAP — Regression Model

Feature importance analysis using SHAP (SHapley Additive exPlanations) on the Optuna-tuned regression model.

In [ ]:
# ── SHAP TreeExplainer — Regression ──────────────────────────────────────
explainer_reg = shap.TreeExplainer(reg_tuned)
shap_values_reg = explainer_reg.shap_values(X_test)

print(f'SHAP values computed for {X_test.shape[0]} test samples, {X_test.shape[1]} features.')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
shap.summary_plot(shap_values_reg, X_test, max_display=20, show=False)
plt.title('SHAP Feature Importance — Regression Model (Top 20)', fontsize=14, fontweight='bold')
plt.tight_layout()
save_fig(plt.gcf(), 'shap_regression_beeswarm')
plt.show()

**Interpretation of Top 5 Features (Regression):**

1. **Lagged notification features** (e.g., `notificacoes_lag4`, `media_movel_4sem`): Recent dengue case counts are the strongest predictors, confirming the autoregressive nature of outbreak dynamics.
2. **Composite epidemiological indices** (e.g., `indice_carga_clinica`, `indice_alarme`): Engineered Gold-layer features that combine symptom proportions and severity signals capture multi-dimensional outbreak patterns.
3. **Seasonal/temporal features** (e.g., `semana_epidemiologica`, `week_of_year_sin`): Dengue follows strong seasonal patterns in Brasilia, peaking during the wet season (weeks 8-16).
4. **Climate variables** (e.g., `temp_mean_c_lag4`, `rain_sum_mm_lag8`): Temperature and rainfall with 4-8 week lags capture the biological cycle of *Aedes aegypti* — precipitation creates breeding sites, and temperature accelerates larval development.
5. **Rate-of-change features** (e.g., `pct_change_4sem`, `aceleracao`): These capture whether the outbreak is accelerating or decelerating, providing early warning signals.

---
## 7. SHAP — Classification Model

In [ ]:
# ── SHAP TreeExplainer — Classification ──────────────────────────────────
explainer_cls = shap.TreeExplainer(cls_tuned)
shap_values_cls = explainer_cls.shap_values(X_test)

# For multiclass, shap_values_cls is a list of arrays (one per class)
# We use mean absolute across classes for the summary
if isinstance(shap_values_cls, list):
    shap_cls_combined = np.mean([np.abs(sv) for sv in shap_values_cls], axis=0)
    print(f'SHAP values computed: {len(shap_values_cls)} classes, {X_test.shape[0]} samples.')
else:
    shap_cls_combined = np.abs(shap_values_cls)
    print(f'SHAP values computed: {X_test.shape[0]} samples.')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

# Use first class or combined for summary plot
if isinstance(shap_values_cls, list):
    # Show Outbreak class (index 3) as most relevant for public health
    shap.summary_plot(shap_values_cls[3], X_test, max_display=20, show=False)
    plt.title('SHAP Feature Importance — Classification (Outbreak Class, Top 20)',
              fontsize=14, fontweight='bold')
else:
    shap.summary_plot(shap_values_cls, X_test, max_display=20, show=False)
    plt.title('SHAP Feature Importance — Classification Model (Top 20)',
              fontsize=14, fontweight='bold')

plt.tight_layout()
save_fig(plt.gcf(), 'shap_classification_beeswarm')
plt.show()

---
## 8. SHAP Dependence Plots

How the top features influence predictions: each point is a test sample, showing the feature value (x-axis) versus its SHAP contribution (y-axis).

In [ ]:
# ── Identify top 5 features by mean |SHAP| (regression) ─────────────────
mean_abs_shap = np.abs(shap_values_reg).mean(axis=0)
top5_indices = np.argsort(mean_abs_shap)[::-1][:5]
top5_features = [feature_names[i] for i in top5_indices]

print('Top 5 features for dependence plots:')
for i, (idx, feat) in enumerate(zip(top5_indices, top5_features), 1):
    print(f'  {i}. {feat} (mean |SHAP| = {mean_abs_shap[idx]:.4f})')

# ── 2x3 subplot grid ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes_flat = axes.flatten()

for i, (feat_idx, feat_name) in enumerate(zip(top5_indices, top5_features)):
    ax = axes_flat[i]
    ax.scatter(
        X_test.iloc[:, feat_idx].values,
        shap_values_reg[:, feat_idx],
        c=X_test.iloc[:, feat_idx].values,
        cmap='coolwarm', s=20, alpha=0.7,
        edgecolors='gray', linewidths=0.2
    )
    ax.axhline(y=0, color='gray', linewidth=0.5, linestyle='-')
    ax.set_xlabel(feat_name, fontsize=9)
    ax.set_ylabel('SHAP value', fontsize=9)
    ax.set_title(f'({chr(97+i)}) {feat_name}', fontsize=10, fontweight='bold')

# Leave last subplot empty with summary text
ax_last = axes_flat[5]
ax_last.axis('off')
summary_text = (
    'SHAP Dependence Plots\n\n'
    'Each point = 1 test sample\n\n'
    'X-axis: feature value\n'
    'Y-axis: SHAP contribution\n'
    'Color: feature value\n\n'
    'Points above 0: feature pushes\n'
    'prediction higher\n\n'
    'Points below 0: feature pushes\n'
    'prediction lower'
)
ax_last.text(0.5, 0.5, summary_text, transform=ax_last.transAxes,
             fontsize=11, verticalalignment='center', horizontalalignment='center',
             bbox=dict(boxstyle='round', facecolor=COLORS['bg_fill'], alpha=0.8))

fig.suptitle('SHAP Dependence — Top 5 Regression Features',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig(fig, 'shap_dependence_top5')
plt.show()

---
## 9. Feature Importance — XGBoost Native vs SHAP

Comparison of XGBoost built-in importance (gain) versus SHAP-based importance. Discrepancies may indicate features that are important for specific subsets of predictions.

In [ ]:
# ── XGBoost native importance (gain) ─────────────────────────────────────
xgb_importance = reg_tuned.feature_importances_

# ── SHAP importance (mean |SHAP|) ────────────────────────────────────────
shap_importance = np.abs(shap_values_reg).mean(axis=0)

# Build comparison DataFrame
importance_df = pd.DataFrame({
    'feature': feature_names,
    'xgb_gain': xgb_importance,
    'shap_mean': shap_importance,
})

# Normalize both to [0, 1] for comparison
importance_df['xgb_norm'] = importance_df['xgb_gain'] / importance_df['xgb_gain'].max()
importance_df['shap_norm'] = importance_df['shap_mean'] / importance_df['shap_mean'].max()

# Rank
importance_df['xgb_rank'] = importance_df['xgb_gain'].rank(ascending=False).astype(int)
importance_df['shap_rank'] = importance_df['shap_mean'].rank(ascending=False).astype(int)
importance_df['rank_diff'] = abs(importance_df['xgb_rank'] - importance_df['shap_rank'])

# Top 15 by either method
top15_xgb = set(importance_df.nsmallest(15, 'xgb_rank')['feature'])
top15_shap = set(importance_df.nsmallest(15, 'shap_rank')['feature'])
top15_union = top15_xgb | top15_shap
top15_df = importance_df[importance_df['feature'].isin(top15_union)].copy()
top15_df = top15_df.sort_values('shap_mean', ascending=True).tail(15)

# ── Side-by-side horizontal bar chart ────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 8))

y_pos = np.arange(len(top15_df))
bar_height = 0.35

bars1 = ax.barh(y_pos + bar_height/2, top15_df['xgb_norm'].values, bar_height,
                label='XGBoost Gain (normalized)', color=COLORS['primary'], alpha=0.8)
bars2 = ax.barh(y_pos - bar_height/2, top15_df['shap_norm'].values, bar_height,
                label='Mean |SHAP| (normalized)', color=COLORS['highlight'], alpha=0.8)

# Highlight features with large rank difference (>5)
for i, (_, row) in enumerate(top15_df.iterrows()):
    if row['rank_diff'] > 5:
        ax.annotate(f'\u0394rank={int(row["rank_diff"])}',
                    xy=(max(row['xgb_norm'], row['shap_norm']) + 0.02, y_pos[i]),
                    fontsize=8, color=COLORS['secondary'], fontweight='bold')

ax.set_yticks(y_pos)
ax.set_yticklabels(top15_df['feature'].values, fontsize=9)
ax.set_xlabel('Normalized Importance', fontsize=11)
ax.set_title('Feature Importance: XGBoost Gain vs SHAP — Top 15',
             fontsize=14, fontweight='bold')
ax.legend(loc='lower right', framealpha=0.9, fontsize=10)
ax.set_xlim(0, 1.15)

plt.tight_layout()
save_fig(fig, 'importance_xgb_vs_shap')
plt.show()

# Print features with largest rank discrepancy
print('\nFeatures with largest rank discrepancy (|XGB rank - SHAP rank| > 5):')
discrepant = importance_df[importance_df['rank_diff'] > 5].nlargest(10, 'rank_diff')
for _, row in discrepant.iterrows():
    print(f'  {row["feature"]}: XGB rank {row["xgb_rank"]}, SHAP rank {row["shap_rank"]} '
          f'(diff={row["rank_diff"]})')

---
## 10. Walk-Forward Cross-Validation Results

Year-by-year temporal validation — each fold trains on all prior years and tests on the next. This tests how well the model generalizes across different dengue seasons.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: Regression R² by year ──────────────────────────────────────────
ax = axes[0]
years_reg = wf_reg['ano'].values
r2_vals = wf_reg['r2_log'].values
mean_r2 = r2_vals.mean()

ax.plot(years_reg, r2_vals, 'o-', color=COLORS['primary'], linewidth=2, markersize=8,
        label='R² (log scale)', zorder=3)
ax.axhline(y=mean_r2, color=COLORS['muted'], linestyle='--', linewidth=1.5,
           label=f'Mean = {mean_r2:.3f}')
ax.fill_between(years_reg, r2_vals, mean_r2, alpha=0.1, color=COLORS['primary'])

# Annotate each point
for year, r2 in zip(years_reg, r2_vals):
    ax.annotate(f'{r2:.2f}', (year, r2), textcoords='offset points',
                xytext=(0, 10), fontsize=8, ha='center', fontweight='bold')

ax.set_xlabel('Test Year', fontsize=11)
ax.set_ylabel('R² (log scale)', fontsize=11)
ax.set_title('(a) Walk-Forward CV — Regression', fontsize=13, fontweight='bold')
ax.legend(loc='lower left', framealpha=0.9)
ax.set_xticks(years_reg)
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%d'))

# ── Right: Classification AUC by year ────────────────────────────────────
ax = axes[1]
years_cls = wf_cls['ano'].values
auc_vals = wf_cls['auc'].values
mean_auc = auc_vals.mean()

ax.plot(years_cls, auc_vals, 'o-', color=COLORS['accent'], linewidth=2, markersize=8,
        label='AUC (macro)', zorder=3)
ax.axhline(y=mean_auc, color=COLORS['muted'], linestyle='--', linewidth=1.5,
           label=f'Mean = {mean_auc:.3f}')
ax.fill_between(years_cls, auc_vals, mean_auc, alpha=0.1, color=COLORS['accent'])

for year, auc_v in zip(years_cls, auc_vals):
    ax.annotate(f'{auc_v:.2f}', (year, auc_v), textcoords='offset points',
                xytext=(0, 10), fontsize=8, ha='center', fontweight='bold')

ax.set_xlabel('Test Year', fontsize=11)
ax.set_ylabel('AUC (macro OvR)', fontsize=11)
ax.set_title('(b) Walk-Forward CV — Classification', fontsize=13, fontweight='bold')
ax.legend(loc='lower left', framealpha=0.9)
ax.set_xticks(years_cls)
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%d'))

fig.suptitle('Walk-Forward Temporal Cross-Validation — Year-by-Year Performance',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
save_fig(fig, 'walk_forward_cv')
plt.show()

# ── Per-year results table ───────────────────────────────────────────────
print('\n=== Walk-Forward CV — Regression ===')
print(wf_reg.to_string(index=False))
print(f'\nMean R²_log: {wf_reg["r2_log"].mean():.4f}')
print(f'Std  R²_log: {wf_reg["r2_log"].std():.4f}')

print('\n=== Walk-Forward CV — Classification ===')
print(wf_cls.to_string(index=False))
print(f'\nMean AUC: {wf_cls["auc"].mean():.4f}')
print(f'Std  AUC: {wf_cls["auc"].std():.4f}')

**Walk-Forward CV Discussion:**

- Walk-forward CV simulates realistic deployment: the model always predicts *future* years using *only past data*, preventing any temporal leakage.
- Year-to-year variance reflects the inherent non-stationarity of dengue dynamics — some seasons have distinctive patterns that the model handles better than others.
- Years with lower R² or AUC often correspond to seasons with unusual outbreak patterns or shifts in reporting practices.
- The mean performance across folds provides a more robust estimate of generalization than a single train/test split.

---
## 11. Distribution Shift Analysis

**Key thesis finding:** The 2024 test period saw a historic dengue outbreak in Brasilia — the worst in recorded history. The training data (up to 2023) had never seen anything close to the 2024 peak. This distribution shift is the primary reason for the low R² on the original scale, while the log-scale R² remains reasonable.

This section quantifies and visualizes the shift.

In [ ]:
# ── Compute annual max notifications for DF ──────────────────────────────
all_data = pd.concat([trainval_df, test_df], ignore_index=True)
annual_max = all_data.groupby('ano')['notificacoes_t4'].max().reset_index()
annual_max.columns = ['Year', 'Max Weekly Notifications']
annual_max = annual_max.sort_values('Year')

# Train max vs test max
train_max = trainval_df['notificacoes_t4'].max()
test_max = test_df['notificacoes_t4'].max()
print(f'Train+Val max (historical): {train_max:,.0f}')
print(f'Test max (2024):            {test_max:,.0f}')
print(f'Ratio: {test_max/train_max:.1f}x')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# ── (a) Annual max notifications bar chart ───────────────────────────────
ax = axes[0]
colors_bar = [COLORS['secondary'] if y >= 2024 else COLORS['primary']
              for y in annual_max['Year']]
bars = ax.bar(annual_max['Year'], annual_max['Max Weekly Notifications'],
              color=colors_bar, alpha=0.85, edgecolor='white', linewidth=0.5)

# Annotate 2024 bar
bar_2024 = annual_max[annual_max['Year'] == 2024]
if len(bar_2024) > 0:
    ax.annotate(
        f'{bar_2024["Max Weekly Notifications"].values[0]:,.0f}\n(historic peak)',
        xy=(2024, bar_2024['Max Weekly Notifications'].values[0]),
        xytext=(2019, bar_2024['Max Weekly Notifications'].values[0] * 0.85),
        fontsize=9, fontweight='bold', color=COLORS['secondary'],
        arrowprops=dict(arrowstyle='->', color=COLORS['secondary'], lw=1.5),
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                  edgecolor=COLORS['secondary'], alpha=0.9),
    )

ax.axhline(y=train_max, color=COLORS['muted'], linestyle='--', linewidth=1,
           label=f'Train max = {train_max:,.0f}')
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Max Weekly Notifications', fontsize=11)
ax.set_title('(a) Annual Peak Dengue — DF', fontsize=12, fontweight='bold')
ax.legend(loc='upper left', fontsize=9, framealpha=0.9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# ── (b) Distribution histogram: train vs test ────────────────────────────
ax = axes[1]
train_target = trainval_df['notificacoes_t4'].dropna()
test_target = test_df['notificacoes_t4'].dropna()

# Use log scale bins for better visualization
bins = np.logspace(0, np.log10(max(test_target.max(), train_target.max()) + 1), 40)
ax.hist(train_target, bins=bins, alpha=0.6, color=COLORS['primary'],
        label=f'Train+Val (n={len(train_target)})', density=True, edgecolor='white')
ax.hist(test_target, bins=bins, alpha=0.6, color=COLORS['secondary'],
        label=f'Test 2024 (n={len(test_target)})', density=True, edgecolor='white')
ax.set_xscale('log')
ax.set_xlabel('Notifications t+4 (log scale)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('(b) Target Distribution Shift', fontsize=12, fontweight='bold')
ax.legend(fontsize=9, framealpha=0.9)

# ── (c) Log-space predictions show model captures trend ──────────────────
ax = axes[2]
ax.scatter(y_test_log, y_pred_log_tuned, alpha=0.6, s=25,
           color=COLORS['primary'], edgecolors='gray', linewidths=0.3)
lim = max(y_test_log.max(), y_pred_log_tuned.max()) * 1.05
ax.plot([0, lim], [0, lim], '--', color=COLORS['secondary'], linewidth=1.5,
        label='Perfect prediction')

# Annotate R²
r2_log_v = r2_score(y_test_log, y_pred_log_tuned)
ax.text(0.05, 0.92, f'R² = {r2_log_v:.3f}', transform=ax.transAxes,
        fontsize=12, fontweight='bold', verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

ax.set_xlabel('Actual — log1p(notifications)', fontsize=11)
ax.set_ylabel('Predicted — log1p(notifications)', fontsize=11)
ax.set_title('(c) Log-Scale Predictions', fontsize=12, fontweight='bold')
ax.legend(loc='lower right', fontsize=9, framealpha=0.9)
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.set_aspect('equal', adjustable='box')

fig.suptitle('Distribution Shift Analysis: 2024 Unprecedented Outbreak in Distrito Federal',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
save_fig(fig, 'distribution_shift_analysis')
plt.show()

**Why R² on original scale is low (~0.06) but log R² is reasonable (~0.46):**

- The 2024 outbreak reached **22,278 weekly notifications** — the training data maximum was only **3,968** (a 5.6x gap).
- On the original scale, a model trained on [0, 3968] simply cannot predict values of 22,000+. The resulting large residuals dominate R², driving it near zero.
- On the **log1p scale**, the gap shrinks from 5.6x to 1.2x (log1p(22278) = 10.01 vs log1p(3968) = 8.29). The model *does* capture the relative trend — it correctly identifies which weeks have higher vs lower notifications.
- This is not a model failure. It is a **distribution shift** — the model learned the correct patterns from historical data but could not extrapolate to an unprecedented scale.
- For public health applications, knowing *when* the outbreak intensifies (relative trend) is arguably more valuable than the exact count, especially during an unprecedented event.

---
## 12. Classification — Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test_cls, y_pred_cls_tuned, labels=[0, 1, 2, 3])
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(8, 7))

# Create annotation strings with count and percentage
annot = np.empty_like(cm, dtype=object)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        count = cm[i, j]
        pct = cm_pct[i, j]
        annot[i, j] = f'{count}\n({pct:.1f}%)'

sns.heatmap(
    cm, annot=annot, fmt='', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    ax=ax, linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'Count'},
    square=True,
)

ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('Actual Label', fontsize=12)
ax.set_title('Confusion Matrix — Risk Level Classification (Test 2024)',
             fontsize=14, fontweight='bold', pad=15)
ax.tick_params(axis='both', labelsize=11)

plt.tight_layout()
save_fig(fig, 'confusion_matrix')
plt.show()

print('\nConfusion matrix (counts):')
cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
cm_df.index.name = 'Actual'
cm_df.columns.name = 'Predicted'
print(cm_df.to_string())

---
## 13. Classification — Per-Class Metrics

In [ ]:
# ── Per-class metrics table ──────────────────────────────────────────────
report = classification_report(
    y_test_cls, y_pred_cls_tuned,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

class_metrics = pd.DataFrame({
    'Class': CLASS_NAMES,
    'Precision': [report[c]['precision'] for c in CLASS_NAMES],
    'Recall': [report[c]['recall'] for c in CLASS_NAMES],
    'F1-Score': [report[c]['f1-score'] for c in CLASS_NAMES],
    'Support': [int(report[c]['support']) for c in CLASS_NAMES],
})

# Add macro/weighted averages
for avg_type in ['macro avg', 'weighted avg']:
    if avg_type in report:
        class_metrics.loc[len(class_metrics)] = [
            avg_type.replace(' avg', ' average').title(),
            report[avg_type]['precision'],
            report[avg_type]['recall'],
            report[avg_type]['f1-score'],
            int(report[avg_type]['support']),
        ]

print('=== Per-Class Classification Metrics ===')
print(class_metrics.to_string(index=False))

# ── Bar chart: precision and recall per class ────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(CLASS_NAMES))
width = 0.3

p_vals = [report[c]['precision'] for c in CLASS_NAMES]
r_vals = [report[c]['recall'] for c in CLASS_NAMES]
f1_vals = [report[c]['f1-score'] for c in CLASS_NAMES]

bars_p = ax.bar(x - width, p_vals, width, label='Precision',
                color=COLORS['primary'], alpha=0.85, edgecolor='white')
bars_r = ax.bar(x, r_vals, width, label='Recall',
                color=COLORS['accent'], alpha=0.85, edgecolor='white')
bars_f = ax.bar(x + width, f1_vals, width, label='F1-Score',
                color=COLORS['highlight'], alpha=0.85, edgecolor='white')

# Annotate bars
for bars in [bars_p, bars_r, bars_f]:
    for bar in bars:
        h = bar.get_height()
        if h > 0.01:
            ax.annotate(f'{h:.2f}', xy=(bar.get_x() + bar.get_width()/2, h),
                        xytext=(0, 4), textcoords='offset points',
                        ha='center', fontsize=8, fontweight='bold')

# Highlight Outbreak class
ax.axvspan(x[-1] - 0.55, x[-1] + 0.55, alpha=0.08, color=COLORS['secondary'])
ax.text(x[-1], -0.08, '(critical for\npublic health)', ha='center', fontsize=8,
        color=COLORS['secondary'], fontstyle='italic')

ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Per-Class Precision, Recall & F1 — Risk Level Classification (Test 2024)',
             fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=10, framealpha=0.9)
ax.set_ylim(0, 1.15)

plt.tight_layout()
save_fig(fig, 'classification_per_class')
plt.show()

---
## 14. SINAN-only vs SINAN+INMET Comparison

Ablation study: does adding INMET climate data improve predictions for the Distrito Federal? Station A001 is located just 1.18 km from Brasilia center, making it an ideal source of local climate data.

In [ ]:
# ── Compute metrics for both models ──────────────────────────────────────
r2_log_full = r2_score(y_test_log, y_pred_log_tuned)
r2_log_sinan = r2_score(y_test_log, y_pred_log_sinan)
r2_orig_full = r2_score(y_test_reg, y_pred_tuned)
r2_orig_sinan = r2_score(y_test_reg, y_pred_sinan)
mae_full = mean_absolute_error(y_test_reg, y_pred_tuned)
mae_sinan = mean_absolute_error(y_test_reg, y_pred_sinan)
rmse_full = np.sqrt(mean_squared_error(y_test_reg, y_pred_tuned))
rmse_sinan = np.sqrt(mean_squared_error(y_test_reg, y_pred_sinan))

# Comparison table
comp_df = pd.DataFrame({
    'Model': ['SINAN+INMET (tuned)', 'SINAN-only'],
    'R2 (log)': [r2_log_full, r2_log_sinan],
    'R2 (original)': [r2_orig_full, r2_orig_sinan],
    'MAE': [mae_full, mae_sinan],
    'RMSE': [rmse_full, rmse_sinan],
    'N features': [len(feature_names), len(sinan_only_cols)],
})

print('=== SINAN+INMET vs SINAN-only ===')
print(comp_df.to_string(index=False))
print(f'\nR2_log improvement: {r2_log_full - r2_log_sinan:+.4f}')
print(f'MAE improvement:    {mae_sinan - mae_full:+.1f} (lower is better)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Left: SINAN+INMET scatter ────────────────────────────────────────────
ax = axes[0]
ax.scatter(y_test_log, y_pred_log_tuned, alpha=0.6, s=25,
           color=COLORS['primary'], edgecolors='gray', linewidths=0.3)
lim = max(y_test_log.max(), y_pred_log_tuned.max(), y_pred_log_sinan.max()) * 1.05
ax.plot([0, lim], [0, lim], '--', color=COLORS['secondary'], linewidth=1.5)
ax.set_xlabel('Actual — log1p(notifications)', fontsize=11)
ax.set_ylabel('Predicted — log1p(notifications)', fontsize=11)
ax.set_title(f'(a) SINAN+INMET — R² = {r2_log_full:.3f}', fontsize=13, fontweight='bold')
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.set_aspect('equal', adjustable='box')

# Add feature count annotation
ax.text(0.05, 0.88, f'{len(feature_names)} features\n(epidemiological + climate)',
        transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle='round', facecolor=COLORS['bg_fill'], alpha=0.9))

# ── Right: SINAN-only scatter ────────────────────────────────────────────
ax = axes[1]
ax.scatter(y_test_log, y_pred_log_sinan, alpha=0.6, s=25,
           color=COLORS['highlight'], edgecolors='gray', linewidths=0.3)
ax.plot([0, lim], [0, lim], '--', color=COLORS['secondary'], linewidth=1.5)
ax.set_xlabel('Actual — log1p(notifications)', fontsize=11)
ax.set_ylabel('Predicted — log1p(notifications)', fontsize=11)
ax.set_title(f'(b) SINAN-only — R² = {r2_log_sinan:.3f}', fontsize=13, fontweight='bold')
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.set_aspect('equal', adjustable='box')

ax.text(0.05, 0.88, f'{len(sinan_only_cols)} features\n(epidemiological only)',
        transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle='round', facecolor='#FFF3E0', alpha=0.9))

fig.suptitle('Ablation: SINAN+INMET vs SINAN-only — Log-Scale Predictions (Test 2024)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig(fig, 'sinan_vs_sinan_inmet')
plt.show()

**Interpretation — SINAN+INMET vs SINAN-only:**

- For the Distrito Federal, adding INMET climate data **improves** predictions. This is expected: station A001 is located only **1.18 km** from Brasilia center, providing highly representative climate measurements.
- Climate features (temperature, rainfall, humidity with 4-8 week lags) capture the biological mechanisms of *Aedes aegypti* mosquito population dynamics.
- The improvement is visible in both R² (log) and MAE, confirming that climate data adds genuine predictive signal rather than overfitting to noise.
- For municipalities with more distant INMET stations, the benefit of climate data may be reduced due to spatial interpolation errors.

---
## 15. Discussion

### Key Findings

1. **XGBoost with log-transformed targets effectively captures dengue outbreak trends** 4 weeks ahead, achieving meaningful R² on the log scale despite the unprecedented 2024 outbreak. The log1p transformation was critical — it reduced the train-test gap from 5.6x to 1.2x.

2. **SHAP analysis confirms domain-relevant features drive predictions**: lagged notification counts, composite epidemiological indices, seasonal patterns, and climate variables (with biologically meaningful lags of 4-8 weeks) are the top predictors.

3. **Climate data improves forecasting for Distrito Federal**: the SINAN+INMET model outperforms the SINAN-only baseline, supported by the proximity of INMET station A001 (1.18 km from Brasilia center).

4. **Walk-forward temporal CV provides robust generalization estimates**: year-by-year validation prevents temporal leakage and reveals performance variation across different dengue seasons.

5. **Distribution shift is the dominant challenge**: the 2024 outbreak was ~5.6x larger than anything in the training data. The model correctly identified outbreak timing and relative magnitude but could not extrapolate to unprecedented absolute counts.

### Limitations

- **Small dataset**: only 1,211 training rows (weekly data for a single municipality). This limits model complexity and generalization.
- **Distribution shift**: models trained on historical data fundamentally cannot predict events outside the training distribution. Periodic retraining with updated data is essential.
- **Single municipality**: results are specific to Distrito Federal (Brasilia). Generalization to other municipalities requires separate validation.
- **Feature lag**: the t+4 prediction horizon means forecasts are 4 weeks ahead, which may be too short for some public health interventions.

### Implications for Public Health

- The model can serve as an **early warning system** — even without predicting exact counts, correctly identifying *when* an outbreak intensifies enables proactive resource allocation.
- The classification model provides interpretable risk levels (Low/Medium/High/Outbreak) that map directly to public health response protocols.
- SHAP-based explanations make the model transparent to epidemiologists, facilitating trust and adoption.

### Future Work

- **Multi-municipality modeling**: extend to all Brazilian capitals with sufficient data.
- **Online learning / retraining**: implement periodic model updates to adapt to distribution shifts.
- **Longer horizons**: explore t+6 and t+8 week predictions for earlier intervention.
- **Additional data sources**: satellite-derived vegetation indices (NDVI), mobility data, and social media signals.

---
## 16. Export All Figures

In [ ]:
import glob as _glob

saved_figs = sorted(_glob.glob(str(OUTPUT_DIR / 'fig_*.png')))

print('=' * 65)
print('EXPORTED FIGURES FOR TCC DOCUMENT')
print('=' * 65)
total_size = 0
for i, path in enumerate(saved_figs, 1):
    p = Path(path)
    size_kb = p.stat().st_size / 1024
    total_size += size_kb
    print(f'  {i:2d}. {p.name:<45s} {size_kb:7.1f} KB')

print(f'\n  Total: {len(saved_figs)} figures, {total_size/1024:.1f} MB')
print(f'  Resolution: 300 DPI')
print(f'  Location: {OUTPUT_DIR}')
print('\nAll figures saved to /kaggle/working/ at 300 DPI, ready for TCC document inclusion.')